In [1]:
import pandas as pd
import numpy as np 
from datetime import datetime, timezone
from typing import Dict, List, Tuple, Optional, Any
import json
import logging
from pathlib import Path
from dataclasses import dataclass, asdict
import hashlib
import re
from sklearn.ensemble import IsolationForest
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Configuration and Schema Definitions

In [2]:
@dataclass
class DataSchema:
    """Define expected schema and business rules"""

    #Column Definitions
    REQUIRED_COLUMNS = [
        'transaction_id', 'timestamp', 'amount', 'currency',
        'merchant_name', 'customer_id', 'channel', 'is_fraud'
    ]

    #Data types
    EXPECTED_DTYPES = {
        'transaction_id': 'object',
        'amount': 'float64',
        'is_fraud': 'int64'
    }

    #Business rules
    AMOUNT_MIN = 0.0
    AMOUNT_MAX = 1000000.0
    VALID_CURRENCIES = {'USD', 'EUR', 'GBP', 'JPY', 'CNY', 'INR', 'AUD', 'CAD'}
    VALID_CHANNELS = {'online', 'in-store', 'mobile', 'atm', 'phone'}
    TIMESTAMP_MIN = '2020-01-01'
    TIMESTAMP_MAX = '2025-12-31'
    
    #Duplicate Detection
    DUPLICATE_WINDOW_SECONDS = 60

    #Outlier Detection
    OUTLIER_CONTAMINATION = 0.05

## Audit Logging System

In [3]:
class AuditLogger:
    """
    Comprehensive audit trail for all data transformations.
    Tracks: original value -> transformed value, rule applied, confidence, timestamps
    """

    def __init__(self):
        self.logs: List[Dict] = []
        self.logger = logging.getLogger(__name__)

    def log_transformation(
        self,
        row_id: Any,
        column: str,
        original_value: Any,
        new_value: Any,
        rule: str,
        confidence: float,
        metadata: Optional[Dict] = None
    ):
        """"Log a single transformation"""
        log_entry = {
            'timestamp': datetime.now(timezone.utc).isoformat(),
            'row_id': row_id,
            'column': column,
            'original_value': str(original_value),
            'new_value': str(new_value),
            'rule': rule,
            'confidence': confidence,
            'metadata': metadata or {}
        }
        self.logs.append(log_entry)

    def get_audit_df(self) -> pd.DataFrame:
        """Return audit log as DataFrame"""
        return pd.DataFrame(self.logs)
    
    def save_audit_log(self, filepath: str):
        """Save audit log to CSV"""
        df = self.get_audit_df()
        df.to_parquet(filepath, index=False)
        self.logger.info(f"Audit log saved to {filepath}")

    def get_summary(self) -> Dict:
        """Get summary statistics of transformations"""
        if not self.logs:
            return {}
        
        df = self.get_audit_df()
        summary = {
            'total_transformations': len(df),
            'columns_affected': df['column'].nunique(),
            'avg_confidence': df['confidence'].mean(),
            'low_confidence_count': (df['confidence'] < 0.7).sum(),
            'rules_applied': df['rule'].value_counts().to_dict()
        }
        return summary

## Data Profiler

In [4]:
class DataProfiler:
    """
    Comprehensive data profiling before any transformations.
    Quantifies missingness, distributions, cardinality, patterns
    """

    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.profile = {}
    
    def generate_profile(self) -> Dict:
        """Generate comprehensive data profile"""
        self.profile = {
            'shape': self.df.shape,
            'memory_usage_mb': self.df.memory_usage(deep=True).sum()/1024**2,
            'columns': self._profile_columns(),
            'duplicates': self._profile_duplicates(),
            'correlations': self._profile_correlations()
        }
        return self.profile
    
    def _profile_columns(self) -> Dict:
        """Profile each column"""
        column_profile = {}

        for col in self.df.columns:
            column_profile[col] = {
                'dtype': str(self.df[col].dtype),
                'missing_count': int(self.df[col].isnull().sum()),
                'missing_pct': float(self.df[col].isnull().mean()*100),
                'unique_count': int(self.df[col].nunique()),
                'unique_pct': float(self.df[col].nunique()/len(self.df)*100),
                'most_common': self._get_most_common(col)
            }
        
        #Numeric Columns
        if pd.api.types.is_numeric_dtype(self.df[col]):
            column_profile[col].update({
                'min': float(self.df[col].min()) if not self.df[col].isnull().all() else None,
                'max': float(self.df[col].max()) if not self.df[col].isnull().all() else None,
                'mean': float(self.df[col].mean()) if not self.df[col].isnull().all() else None,
                'median': float(self.df[col].median()) if not self.df[col].isnull().all() else None,
                'std': float(self.df[col].std()) if not self.df[col].isnull().all() else None,
                'zeros_count': int((self.df[col] == 0).sum()),
                'negative_count': int((self.df[col] < 0).sum()) if not self.df[col].isnull().all() else None
            })

        return column_profile
    
    def _get_most_common(self, series: pd.Series, n: int = 5) -> List:
        """Get most common values"""
        try:
            return series.value_counts().head(n).to_dict()
        except:
            return {}
    
    def _profile_duplicates(self) -> Dict:
        """Profile duplicate rows"""
        return {
            'duplicate_rows': int(self.df.duplicated().sum()),
            'duplicate_pct': float(self.df.duplicated().mean()*100)
        }
    
    def _profile_correlations(self) -> Dict:
        """Profile numeric correlations"""
        numeric_cols = self.df.select_dtypes(include=[np.number]).columns
        if(len(numeric_cols) > 1):
            corr_matrix = self.df[numeric_cols].corr()
            return {
                'high_correlations': self._find_high_correlations(corr_matrix)
            }
        
        return {}
    
    def _find_high_correlations(self, corr_matrix: pd.DataFrame, threshold: float = 0.8) -> List:
        """Find highly correlated pairs"""
        high_corr = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                if abs(corr_matrix.iloc[i, j]) > threshold:
                    high_corr.append({
                        'col1': corr_matrix.columns[i],
                        'col2': corr_matrix.columns[j],
                        'correlation': float(corr_matrix.iloc[i, j])
                    })
        return high_corr
    
    def save_profile(self, filepath: str):
        """Save profile to JSON"""
        with open(filepath, 'w') as f:
            json.dump(self.profile, f, indent=2)

## Time Stamp Normalizer

In [ ]:
class TimestampNormalizer:
    """
    Robust timestamp parsing and normalization.
    Handles multiple formats, timezones and ambiguous dates.
    """

    COMMON_FORMATS = [
        '%Y-%m-%d %H:%M:%S',
        '%Y-%m-%d %H:%M:%S.%f',
        '%Y/%m/%d %H:%M:%S',
        '%d-%m-%Y %H:%M:%S',
        '%m-%d-%Y %H:%M:%S',
        '%Y-%m-%dT%H:%M:%S',
        '%Y-%m-%dT%H:%M:%SZ',
        '%Y-%m-%dT%H:%M:%S.%fZ',
        '%d/%m/%Y %H:%M:%S',
        '%m/%d/%Y %H:%M:%S',
    ]

    def __init__(self, audit_logger: AuditLogger):
        self.audit_logger = audit_logger

    def parse_timestamp(
        self,
        row_id: Any, 
        timestamp_str: Any,
        default_tz: timezone = timezone.utc
    ) -> Tuple[Optional[pd.Timestamp], float]:
        """
        Parse timestamp with confidence scoring
        Returns: (parsed_timestamp, confidence)
        """

        if pd.isna(timestamp_str):
            return None, 0.0
        
        if isinstance(timestamp_str, (pd.Timestamp, datetime)):
            return pd.Timestamp(timestamp_str, tz=default_tz), 1.0
        timestamp_str = str(timestamp_str).strip()
        
        #Try common formats
        for fmt in self.COMMON_FORMATS:
            try:
                dt = datetime.strptime(timestamp_str, fmt)
                #Add timezone if naive
                if dt.tzinfo is None:
                    dt = dt.replace(tzinfo=default_tz)
                confidence = 0.95
                return pd.Timestamp(dt), confidence
            except:
                continue
        
        #Try pandas flexible parsing
        try:
            dt = pd.to_datetime(timestamp_str)
            if dt.tzinfo is None:
                dt = dt.tz_localize(default_tz)
            confidence = 0.0
            return dt, confidence
        except:
            pass

        return None, 0.0 
    
    def normalize_timestamps(
        self,
        df: pd.DataFrame,
        timestamp_col: str = 'timestamp',
    ) -> pd.DataFrame:
        """Normalize all timestamps in the dataframe"""

        df = df.copy()
        parsed_timestamps = []
        confidences = []

        for idx, val in df[timestamp_col].items():
            parsed, conf = self.parse_timestamp(idx, val)
            parsed_timestamps.append(parsed)
            confidences.append(conf)

            #log transformation
            if parsed is not None and str(val) != str(parsed):
                self.audit_logger.log_transformation(
                    row_id = idx,
                    column = timestamp_col,
                    original_value = val,
                    new_value = parsed,
                    rule = 'timestamp_parsing',
                    confidence = conf
                )
        df[timestamp_col] = parsed_timestamps
        df[f'{timestamp_col}_confidence'] = confidences

        return df

## Amount and Currency Normalization

In [6]:
class AmountCurrencyNormalizer:
    """
    Parse and normalize amounts and currencies
    Handles: symbols ($, €, £), word amounts, different formats.
    """

    CURRENCY_SYMBOLS = {
        '$': 'USD', '€': 'EUR', '£': 'GBP', '¥': 'JPY',
        '₹': 'INR', 'A$': 'AUD', 'C$': 'CAD'
    }

    #Simplified exchange rates
    EXCHANGE_RATES = {
        'USD': 1.0, 'EUR': 1.08, 'GBP': 1.27, 'JPY': 0.0067,
        'CNY': 0.14, 'INR': 0.012, 'AUD': 0.65, 'CAD': 0.73
    }

    def __init__(self, audit_logger: AuditLogger):
        self.audit_logger = audit_logger
    
    def parse_amount(
        self,
        row_id: Any,
        amount_str: Any,
    ) -> Tuple[Optional[float], float]:
        """
        Parse amount with confidence scoring
        Returns: (parsed_amount, confidence)
        """

        if pd.isnull(amount_str):
            return None, 0.0
        
        #Already numeric
        if isinstance(amount_str, (int, float)):
            return float(amount_str), 1.0
        
        amount_str = str(amount_str).strip()

        #Remove currency symbols
        cleaned = re.sub(r'[,$£€¥₹]', '', amount_str)
        cleaned = cleaned.replace(' ', '')

        try:
            amount = float(cleaned)
            confidence = 0.95 if amount >= 0 else 0.5
            return amount, confidence
        except:
            pass

        #Try to extract numbers
        numbers = re.findall(r'\d+\.?\d*', amount_str)
        if numbers:
            try:
                amount = float(numbers[0])
                return amount, 0.7
            except:
                pass
        
        return None, 0.0
    
    def parse_currency(
        self,
        row_id: Any,
        currency_str: Any
    ) -> Tuple[Optional[str], float]:
        """
        Parse currency with confidence scoring
        Returns: (parsed_currency, confidence)
        """

        if pd.isnull(currency_str):
            return None, 0.0
        
        currency_str = str(currency_str).strip().upper()

        #Direct match
        if currency_str in DataSchema.VALID_CURRENCIES:
            return currency_str, 1.0

        #Symbol match
        for symbol, code in self.CURRENCY_SYMBOLS.items():
            if symbol in str(currency_str):
                return code, 0.9
        
        #Fuzzy match
        for valid_curr in DataSchema.VALID_CURRENCIES:
            if valid_curr in currency_str or currency_str in valid_curr:
                return valid_curr, 0.7
        
        return None, 0.0
    
    def normalize_amounts_currencies(
        self,
        df: pd.DataFrame,
        amount_col: str = 'amount',
        currency_col: str = 'currency'
    ) -> pd.DataFrame:
        """Normalize amounts and currencies"""
        df = df.copy()

        #Parse amounts
        parsed_amounts = []
        amount_confidences = []
        for idx, val in df[amount_col].items():
            parsed, conf = self.parse_amount(idx, val)
            parsed_amounts.append(parsed)
            amount_confidences.append(conf)

            if parsed is not None and val != parsed:
                self.audit_logger.log_transformation(
                    row_id = idx,
                    column = amount_col,
                    original_value = val,
                    new_value = parsed,
                    rule = 'amount_parsing',
                    confidence = conf
                )
        
        df[amount_col] = parsed_amounts
        df[f'{amount_col}_confidence'] = amount_confidences

        #Parse currencies
        parsed_currencies = []
        currency_confidences = []
        for idx, val in df[currency_col].items():
            parsed, conf = self.parse_currency(idx, val)
            parsed_currencies.append(parsed)
            currency_confidences.append(conf)
            if parsed is not None and val != parsed:
                self.audit_logger.log_transformation(
                    row_id = idx,
                    column = currency_col,
                    original_value = val,
                    rule = 'currency_parsing',
                    confidence = conf
                )
        
        df[currency_col] = parsed_currencies
        df[f'{currency_col}_confidence'] = currency_confidences

        # Convert to USD
        df['amount_usd'] = df.apply(
            lambda row: self._convert_to_usd(row[amount_col], row[currency_col]),
            axis = 1
        )

        return df
    
    def _convert_to_usd(
        self,
        amount: Optional[float],
        currency: Optional[str]
    ) -> Optional[float]:
        """Convert amount to USD"""

        if pd.isnull(amount) or pd.isnull(currency):
            return None
        if currency not in self.EXCHANGE_RATES:
            return None
        
        return amount * self.EXCHANGE_RATES[currency]


## Categorical Canonicalization

In [7]:
class CategoricalCanonicalizer:
    """
    Canonicalize categorical text fields(merchants, channels, etc)
    Uses fuzzy matching and learned embeddings
    """

    def __init__(self, audit_logger: AuditLogger):
        self.audit_logger = audit_logger
        self.canonical_mappings = {}
    
    def build_canonical_mapping(
        self, 
        df: pd.DataFrame,
        column: str,
        valid_values: Optional[set] = None,
        min_frequency: int = 5
    ) -> Dict[str, str]:
        """
        Build canonical mapping for a categorical column.
        Groups similar values and maps to most common variant
        """

        if column not in df.columns:
            return {}
        
        #Get value counts
        value_counts = df[column].dropna().value_counts()

        #Filter by frequency
        frequent_values = value_counts[value_counts >= min_frequency].index.tolist()

        mappings = {}

        if valid_values:
            # Map to valid values
            for val in frequent_values:
                canonical = self._find_best_match(val, valid_values)
                mappings[val] = canonical
        else:
            #Group similar values
            groups = self._group_similar_values(frequent_values)
            for group in groups:
                canonical = max(group, key=lambda x: value_counts.get(x, 0))
                for val in group:
                    mappings[val] = canonical
        
        self.canonical_mappings[column] = mappings
        return mappings
    
    def _find_best_match(self, value: str, valid_values: set) -> str:
        """Finding the best match valid value"""

        value_clean = str(value).lower().strip()
        #Exact match
        for valid in valid_values:
            if(value_clean == str(valid).lower().strip()):
                return valid
        
        #Fuzzy match
        for valid in valid_values:
            if value_clean in str(valid).lower().strip() or str(valid).lower().strip() in value_clean:
                return valid
        
        #Return original value if no match
        return value
    
    def _group_similar_values(self, values: List[str], threshold: float=0.85) -> List[List[str]]:
        """Group similar values using simple similarity"""

        #Simplified grouping
        groups = []
        processed = set()

        for val in values:
            if val in processed:
                continue

            group = [val]
            val_clean = str(val).lower().strip()

            for other in values:
                if other == val or other in processed:
                    continue

                other_clean = str(other).lower().strip()

                #Simple similarity check
                if (val_clean in other_clean or other_clean in val_clean or
                    self._simple_similarity(val_clean, other_clean) > threshold):
                    group.append(other)
                    processed.add(other)
            
            groups.append(group)
            processed.add(val)
        
        return groups
    
    def _simple_similarity(self, s1: str, s2: str) -> float:
        """Simple character-based similarity"""

        if not s1 or not s2:
            return 0.0
        
        s1_chars = set(s1)
        s2_chars = set(s2)
        intersection = len(s1_chars & s2_chars)
        union = len(s1_chars | s2_chars)
        return intersection / union if union > 0 else 0.0

    def apply_canonical_mapping(
        self,
        df: pd.DataFrame,
        column: str
    ) -> pd.DataFrame:
        """Apply canonical mapping to column"""
        if column not in self.canonical_mappings:
            return df
        
        df = df.copy()
        mapping = self.canonical_mappings[column]

        for idx, val in df[column].items():
            if pd.notna(val) and val in mapping:
                new_val = mapping[val]
                if new_val != val:
                    self.audit_logger.log_transformation(
                        row_id = idx,
                        column = column,
                        original_value = val,
                        new_value = new_val,
                        rule = 'canonical_mapping',
                        confidence = 0.9
                    )
                    df.at[idx, column] = new_val
        
        return df

## Duplicate Detection and Resolution

In [8]:
class DuplicateHandler:
    """
    Sophisticated duplicate detection using multiple strategies
    """

    def __init__(self, audit_logger: AuditLogger):
        self.audit_logger = audit_logger
    
    def generate_row_signature(self, row: pd.Series, key_columns: List[str]) -> str:
        """Generate hash signature for duplicate detection"""
        values = [str(row[col]) for col in key_columns if col in row.index]
        signature_str = '|'.join(values)
        return hashlib.md5(signature_str.encode()).hexdigest()
        
    def detect_exact_duplicates(self, df: pd.DataFrame, subset: Optional[List[str]] = None) -> pd.DataFrame:
        """Detect exact duplicate rows"""

        df = df.copy()
        df['is_exact_duplicate'] = df.duplicated(subset=subset, keep = 'first')
        
        #log duplicates
        dup_indices = df[df['is_exact_duplicate']].index
        for idx in dup_indices:
            self.audit_logger.log_transformation(
                row_id = idx,
                column = '_row',
                original_value = 'keep',
                new_value = 'duplicate',
                rule = 'exact_duplicate_detection',
                confidence = 1.0
            )
        return df
    
    def detect_fuzzy_duplicates(
        self,
        df: pd.DataFrame, 
        key_columns: List[str],
        time_window_seconds: int =60
    ) -> pd.DataFrame:
        """
        Detect fuzzy duplicates: same key fields within time window
        Common in fraud detection - same transaction submitted multiple times
        """

        df = df.copy()
        df['signature'] = df.apply(
            lambda row: self.generate_row_signature(row, key_columns),
            axis = 1
        )
        
        #Sort by timestamp and signature
        if 'timestamp' in df.columns:
            df = df.sort_values(['signature'], 'timestamp')

            #Identify fuzzy duplicates
            df['is_fuzzy_duplicate'] = False
            for sig in df['signature'].unique():
                sig_mask = df['signature'] == sig
                sig_df = df[sig_mask].copy()

                if len(sig_df) > 1:
                    #Check time windows
                    for i in range(1, len(sig_df)):
                        time_diff = (sig_df.iloc[i]['timestamp']-sig[i-1]['timestamp']).total_seconds()
                        if abs(time_diff) <= time_window_seconds:
                            df.loc[sig_df.iloc[i].name, 'is_fuzzy_duplicate'] = True
        
        return df
    
    def resolve_duplicates(
        self,
        df: pd.DataFrame,
        strategy: str = 'keep_first'
    ) -> pd.DataFrame:
        """
        Resolve duplicates based on strategy.
        """
        df = df.copy()
        # Mark duplicates for removal
        duplicate_cols = [c for c in df.columns if 'duplicate' in c.lower()]
        if duplicate_cols:
            mask = df[duplicate_cols].any(axis=1)
            df['_to_remove'] = mask
        
        return df

## Outlier and Anomaly Detection

In [9]:
class OutlierDetector:
    """
    Multi-method outlier detection: Statistical + ML based
    """

    def __init__(self, audit_logger: AuditLogger):
        self.audit_logger = audit_logger
    
    def detect_statistical_outliers(
        self,
        df: pd.DataFrame,
        column: str,
        method: str = 'mad',
        threshold: float = 3.5
    ) -> pd.Series:
        """
        Detect outliers using statistical methods
        Methods: MAD(Median Absolute Deviation), IQR, Z-score
        """

        if column not in df.columns or df[column].isnull().all():
            return pd.Series([False]*len(df), index=df.index)
        
        values = df[column].dropna()

        if method.lower().strip() == 'mad':
            median = values.median()
            mad = np.median(np.abs(values-median))
            if mad == 0:
                return pd.Series([False]*len(df), index=df.index)
            modified_z_scores = 0.6745*(df[column]-median)/mad
        
        elif method.lower().strip() == 'iqr':
            Q1 = values.quantile(0.25)
            Q3 = values.quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5*IQR
            upper = Q3 + 1.5*IQR
            return (df[column] < lower) | (df[column] > upper)
        
        elif method.lower().strip() == 'z-score':
            z_scores = np.abs(stats.zscore(values, nan_policy='omit'))
            outlier_mask = pd.Series([False]*len(df), index=df.index)
            outlier_mask.loc[values.index] = z_scores > threshold
            return outlier_mask        

        return pd.Series([False]*len(df), index=df.index)
    
    def detect_ml_outliers(
        self,
        df: pd.DataFrame,
        feature_columns: List[str],
        contamination: float = 0.05
    ) -> pd.Series:
        """Detect outliers using Isolation Forest"""
        #Select numerical features
        features = df[feature_columns].select_dtypes(include=[np.number])

        if features.empty or features.shape[1] == 0:
            return pd.Series([False]*len(df), index=df.index)
        
        #Handle missing values
        features_filled = features.fillna(features.median())

        #Fit isolation forest
        iso_forest = IsolationForest(
            contamination = contamination,
            random_state = 42,
            n_estimators = 100
        )

        predictions = iso_forest.fit_predict(features_filled)
        outlier_mask = predictions == -1

        return pd.Series(outlier_mask, index=df.index)
    
    def detect_all_outliers(
        self,
        df: pd.DataFrame,
        numeric_columns: Optional[List[str]] = None
    ) -> pd.DataFrame:
        """Detect outliers using multiple methods"""
        df = df.copy()

        if numeric_columns is None:
            numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
        
        #Statistical outliers per column
        for col in numeric_columns:
            if col in df.columns:
                outlier_mask = self.detect_statistical_outliers(df, col, method='mad')
                df[f'{col}_is_outlier'] = outlier_mask

                #Log outliers
                outlier_indices = df[outlier_mask].index
                for idx in outlier_indices:
                    self.audit_logger.log_transformation(
                        row_id = idx,
                        column = col,
                        original_value = df.at[idx, col],
                        new_value = 'OUTLIER_DETECTED',
                        rule = 'mad_outlier_detection',
                        confidence = 0.85
                    )
        
        #ML-based outliers
        if len(numeric_columns) > 1:
            ml_outliers = self.detect_ml_outliers(df, numeric_columns)
            df['ml_outliers'] = ml_outliers
        
        return df

## Missingness Strategy

In [10]:
class MissingnessHandler:
    """
    Strategic imputation based on data type and patterns
    """

    def __init__(self, audit_logger: AuditLogger):
        self.audit_logger = audit_logger
        self.imputation_values = {}
    
    def analyze_missingness(self, df: pd.DataFrame) -> Dict:
        """Analyze missingness patterns"""
        missing_analysis = {}

        for col in df.columns:
            missing_count = df[col].isnull().sum()
            if missing_count > 0:
                missing_analysis[col] = {
                    'count': int(missing_count),
                    'percentage': float(missing_count/len(df)*100),
                    'pattern': self._detect_missing_pattern(df, col)
                }
        return missing_analysis
    
    def _detect_missing_pattern(self, df: pd.DataFrame, col:str) -> str:
        """
        Detects is missingness is random or not
        """
        missing_mask = df[col].isnull()

        #Simple heuristic: if >80% missing, likely systematic
        missing_pct = missing_mask.mean()
        if missing_pct > 0.8:
            return 'systematic'
        elif missing_pct < 0.05:
            return 'random'
        else:
            return 'mixed'
        
    def impute_column(
        self, 
        df: pd.DataFrame,
        column: str,
        strategy: str = 'auto'
    ) -> pd.DataFrame:
        """
        Impute missing values with appropriate strategy
        Strategies: auto, median, mode, forward_fill, flag
        """

        df = df.copy()

        if column not in df.columns:
            return df
        
        missing_mask = df[column].isnull()
        if not missing_mask.any():
            return df
        
        if strategy == 'auto':
            if pd.api.types.is_numeric_dtype(df[column]):
                strategy = 'median'
            else:
                strategy = 'mode'
        
        #Apply imputation
        if strategy == 'median':
            fill_value = df[column].median()
            df.loc[missing_mask, column] = fill_value
            self.imputation_values[column] = fill_value
            confidence = 0.7
        
        elif strategy == 'mode':
            fill_value = df[column].mode()[0] if not df[column].mode().empty else 'UNKNOWN'
            df.loc[missing_mask, column] = fill_value
            self.imputation_values[column] = fill_value
            confidence = 0.6
        
        elif strategy == 'forward_fill':
            df[column] = df[column].fillna(method='ffill')
        
        elif strategy == 'flag':
            #Create missing indicator
            df[f'{column}_was_missing'] = missing_mask
            df[column] = df[column].fillna('MISSING')
            confidence = 0.8
        
        imputed_indices = missing_mask[missing_mask].index
        for idx in imputed_indices:
            self.audit_logger.log_transformation(
                row_id=idx,
                column=column,
                original_value=None,
                new_value=df.at[idx, column],
                rule=f'imputation_{strategy}',
                confidence=confidence
            )

        return df

## Validation and Great Expectations

In [11]:
class DataValidator:
    """
    Validate data against business rules and schema.
    Implements Great Expectations-style validations.
    """
    
    def __init__(self, schema: DataSchema):
        self.schema = schema
        self.validation_results = []
        
    def validate_all(self, df: pd.DataFrame) -> Dict:
        """Run all validations"""
        results = {
            'schema_validation': self.validate_schema(df),
            'business_rules': self.validate_business_rules(df),
            'data_quality': self.validate_data_quality(df),
            'summary': {}
        }
        
        # Aggregate results
        total_checks = sum(len(v['checks']) for v in results.values() if isinstance(v, dict) and 'checks' in v)
        passed_checks = sum(
            sum(1 for c in v['checks'] if c['passed']) 
            for v in results.values() 
            if isinstance(v, dict) and 'checks' in v
        )
        
        results['summary'] = {
            'total_checks': total_checks,
            'passed_checks': passed_checks,
            'failed_checks': total_checks - passed_checks,
            'pass_rate': passed_checks / total_checks if total_checks > 0 else 0
        }
        
        return results
    
    def validate_schema(self, df: pd.DataFrame) -> Dict:
        """Validate schema requirements"""
        checks = []
        
        # Check required columns
        for col in self.schema.REQUIRED_COLUMNS:
            passed = col in df.columns
            checks.append({
                'check': f'column_exists_{col}',
                'passed': passed,
                'message': f'Column {col} exists' if passed else f'Missing column: {col}'
            })
        
        # Check data types
        for col, expected_dtype in self.schema.EXPECTED_DTYPES.items():
            if col in df.columns:
                actual_dtype = str(df[col].dtype)
                passed = expected_dtype in actual_dtype or actual_dtype in expected_dtype
                checks.append({
                    'check': f'dtype_{col}',
                    'passed': passed,
                    'message': f'{col} dtype: {actual_dtype} (expected: {expected_dtype})'
                })
        
        return {'checks': checks}
    
    def validate_business_rules(self, df: pd.DataFrame) -> Dict:
        """Validate business rules / invariants"""
        checks = []
        
        # Amount >= 0
        if 'amount' in df.columns:
            negative_count = (df['amount'] < 0).sum()
            passed = negative_count == 0
            checks.append({
                'check': 'amount_non_negative',
                'passed': passed,
                'message': f'{negative_count} negative amounts found' if not passed else 'All amounts non-negative'
            })
            
            # Amount within reasonable range
            max_amount = df['amount'].max()
            passed = max_amount <= self.schema.AMOUNT_MAX
            checks.append({
                'check': 'amount_within_max',
                'passed': passed,
                'message': f'Max amount: {max_amount}'
            })
        
        # Valid currencies
        if 'currency' in df.columns:
            invalid_currencies = df[~df['currency'].isin(self.schema.VALID_CURRENCIES)]['currency'].unique()
            passed = len(invalid_currencies) == 0
            checks.append({
                'check': 'valid_currencies',
                'passed': passed,
                'message': f'Invalid currencies: {invalid_currencies}' if not passed else 'All currencies valid'
            })
        
        # Valid channels
        if 'channel' in df.columns:
            invalid_channels = df[~df['channel'].isin(self.schema.VALID_CHANNELS)]['channel'].unique()
            passed = len(invalid_channels) == 0
            checks.append({
                'check': 'valid_channels',
                'passed': passed,
                'message': f'Invalid channels: {invalid_channels}' if not passed else 'All channels valid'
            })
        
        # Timestamp range
        if 'timestamp' in df.columns:
            min_ts = pd.Timestamp(self.schema.TIMESTAMP_MIN)
            max_ts = pd.Timestamp(self.schema.TIMESTAMP_MAX)
            out_of_range = ((df['timestamp'] < min_ts) | (df['timestamp'] > max_ts)).sum()
            passed = out_of_range == 0
            checks.append({
                'check': 'timestamp_range',
                'passed': passed,
                'message': f'{out_of_range} timestamps out of range' if not passed else 'All timestamps in range'
            })
        
        # Unique transaction IDs
        if 'transaction_id' in df.columns:
            duplicates = df['transaction_id'].duplicated().sum()
            passed = duplicates == 0
            checks.append({
                'check': 'unique_transaction_ids',
                'passed': passed,
                'message': f'{duplicates} duplicate transaction IDs' if not passed else 'All transaction IDs unique'
            })
        
        return {'checks': checks}
    
    def validate_data_quality(self, df: pd.DataFrame) -> Dict:
        """Validate overall data quality"""
        checks = []
        
        # Check for excessive missing values
        for col in df.columns:
            missing_pct = df[col].isna().mean() * 100
            passed = missing_pct < 50  # Arbitrary threshold
            if not passed:
                checks.append({
                    'check': f'missing_threshold_{col}',
                    'passed': passed,
                    'message': f'{col}: {missing_pct:.1f}% missing (>50%)'
                })
        
        # Check for low cardinality in expected high-cardinality columns
        high_card_cols = ['transaction_id', 'customer_id', 'merchant_name']
        for col in high_card_cols:
            if col in df.columns:
                unique_pct = df[col].nunique() / len(df) * 100
                passed = unique_pct > 10  # At least 10% unique
                checks.append({
                    'check': f'cardinality_{col}',
                    'passed': passed,
                    'message': f'{col}: {unique_pct:.1f}% unique'
                })
        
        return {'checks': checks}
    
    def save_validation_report(self, results: Dict, filepath: str):
        """Save validation results"""
        with open(filepath, 'w') as f:
            json.dump(results, f, indent=2, default=str)

## Gold Test Set Generator

In [13]:
class GoldTestSetGenerator:
    """
    Generate gold test for measuring cleaning quality.
    Two modes: manual correction storage and synthetic corruption
    """

    def __init__(self):
        self.gold_data = []
    
    def create_from_corruptions(
        self,
        clean_df: pd.DataFrame,
        n_samples: int = 1000,
        corruption_rate: float = 0.3
    ) -> Tuple[pd.DataFrame, pd.DataFrame]:
        """
        Create gold test set by corrupting clean data.
        Returns: (corrupted_df, ground_truth_df)
        """
        # Sample rows
        if len(clean_df) < n_samples:
            n_samples = len(clean_df)
        
        sample_df = clean_df.sample(n=n_samples, random_state=42).copy()
        ground_truth = sample_df.copy()
        corrupted = sample_df.copy()
        
        # Apply corruptions
        n_corruptions = int(len(sample_df) * corruption_rate)
        corruption_indices = np.random.choice(sample_df.index, size=n_corruptions, replace=False)
        
        for idx in corruption_indices:
            corruption_type = np.random.choice([
                'timestamp_format', 'amount_format', 'currency_symbol',
                'merchant_typo', 'missing_value'
            ])
            
            if corruption_type == 'timestamp_format':
                # Corrupt timestamp format
                if 'timestamp' in corrupted.columns:
                    ts = corrupted.at[idx, 'timestamp']
                    # Random format corruption
                    corrupted.at[idx, 'timestamp'] = ts.strftime('%d/%m/%Y %H:%M:%S')
            
            elif corruption_type == 'amount_format':
                # Add currency symbol to amount
                if 'amount' in corrupted.columns:
                    amount = corrupted.at[idx, 'amount']
                    corrupted.at[idx, 'amount'] = f'${amount:,.2f}'
            
            elif corruption_type == 'currency_symbol':
                # Replace currency code with symbol
                if 'currency' in corrupted.columns:
                    currency = corrupted.at[idx, 'currency']
                    symbol_map = {'USD': '$'
        , 'EUR': '€', 'GBP': '£'}
                    if currency in symbol_map:
                        corrupted.at[idx, 'currency'] = symbol_map[currency]
            
            elif corruption_type == 'merchant_typo':
                # Add typo to merchant name
                if 'merchant_name' in corrupted.columns:
                    merchant = str(corrupted.at[idx, 'merchant_name'])
                    if len(merchant) > 3:
                        pos = np.random.randint(0, len(merchant))
                        merchant_list = list(merchant)
                        merchant_list[pos] = 'X'
                        corrupted.at[idx, 'merchant_name'] = ''.join(merchant_list)
            
            elif corruption_type == 'missing_value':
                # Introduce missing value
                cols = ['merchant_name', 'channel', 'currency']
                col = np.random.choice([c for c in cols if c in corrupted.columns])
                corrupted.at[idx, col] = np.nan
        
        return corrupted, ground_truth
    
    def evaluate_cleaning_pipeline(
        self,
        cleaned_df: pd.DataFrame,
        ground_truth_df: pd.DataFrame,
        columns_to_check: Optional[List[str]] = None
    ) -> Dict:
        """
        Evaluate cleaning quality against ground truth.
        Returns metrics: accuracy, precision, recall
        """

        if columns_to_check is None:
            columns_to_check = ground_truth_df.columns.tolist()
        
        metrics = {}

        for col in columns_to_check:
            if col not in cleaned_df.columns or col not in ground_truth_df.columns:
                continue

            #Align indices
            common_idx = cleaned_df.index.intersection(ground_truth_df.index)
            if len(common_idx) == 0:
                continue
            
            cleaned_vals = cleaned_df.loc[common_idx, col]
            truth_vals = ground_truth_df.loc[common_idx, col]

            #Calculate accuracy
            matches = (cleaned_vals == truth_vals) | (cleaned_vals.isna() & truth_vals.isna())
            accuracy = matches.mean()

            metrics[col] = {
                'accuracy': float(accuracy),
                'total_values': len(common_idx),
                'correct_values': int(matches.sum())
            }
        
        #Overall metrics
        overall_accuracy = np.mean([m['accuracy'] for m in metrics.values()])
        metrics['overall'] = {
            'accuracy': float(overall_accuracy),
            'columns_evaluated': len(metrics)-1
        }

        return metrics

In [14]:
class DataCleaningPipeline:
    """
    Main orchestration pipeline that ties all components together.
    Implements reversible, auditable, production-grade data cleaning.
    """
    
    def __init__(self, config: Optional[Dict] = None):
        self.config = config or {}
        self.audit_logger = AuditLogger()
        self.schema = DataSchema()
        
        # Initialize components
        self.profiler = None
        self.timestamp_normalizer = TimestampNormalizer(self.audit_logger)
        self.amount_normalizer = AmountCurrencyNormalizer(self.audit_logger)
        self.categorical_canonicalizer = CategoricalCanonicalizer(self.audit_logger)
        self.duplicate_handler = DuplicateHandler(self.audit_logger)
        self.outlier_detector = OutlierDetector(self.audit_logger)
        self.missingness_handler = MissingnessHandler(self.audit_logger)
        self.validator = DataValidator(self.schema)
        
        # Store original data
        self.original_df = None
        self.cleaned_df = None
        self.profile_results = None
        self.validation_results = None
        
    def fit(self, df: pd.DataFrame) -> 'DataCleaningPipeline':
        """
        Fit the pipeline on training data.
        Learns mappings, imputation values, etc.
        """
        logging.info("Starting pipeline fitting...")
        
        # Store original
        self.original_df = df.copy()
        
        # Profile data
        logging.info("Step 1/10: Profiling data...")
        self.profiler = DataProfiler(df)
        self.profile_results = self.profiler.generate_profile()
        
        # Build canonical mappings
        logging.info("Step 2/10: Building canonical mappings...")
        if 'channel' in df.columns:
            self.categorical_canonicalizer.build_canonical_mapping(
                df, 'channel', valid_values=self.schema.VALID_CHANNELS
            )
        
        if 'merchant_name' in df.columns:
            self.categorical_canonicalizer.build_canonical_mapping(
                df, 'merchant_name'
            )
        
        logging.info("Pipeline fitting complete.")
        return self
    
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Transform data through the cleaning pipeline.
        All operations are logged and reversible.
        """
        logging.info("Starting data transformation...")
        
        df_clean = df.copy()
        
        # Step 1: Normalize timestamps
        logging.info("Step 3/10: Normalizing timestamps...")
        if 'timestamp' in df_clean.columns:
            df_clean = self.timestamp_normalizer.normalize_timestamps(df_clean)
        
        # Step 2: Normalize amounts and currencies
        logging.info("Step 4/10: Normalizing amounts and currencies...")
        if 'amount' in df_clean.columns and 'currency' in df_clean.columns:
            df_clean = self.amount_normalizer.normalize_amounts_currencies(df_clean)
        
        # Step 3: Canonicalize categorical fields
        logging.info("Step 5/10: Canonicalizing categorical fields...")
        for col in ['channel', 'merchant_name']:
            if col in df_clean.columns:
                df_clean = self.categorical_canonicalizer.apply_canonical_mapping(df_clean, col)
        
        # Step 4: Detect duplicates
        logging.info("Step 6/10: Detecting duplicates...")
        df_clean = self.duplicate_handler.detect_exact_duplicates(df_clean)
        if 'timestamp' in df_clean.columns:
            df_clean = self.duplicate_handler.detect_fuzzy_duplicates(
                df_clean,
                key_columns=['customer_id', 'amount', 'merchant_name']
            )
        
        # Step 5: Detect outliers
        logging.info("Step 7/10: Detecting outliers...")
        numeric_cols = ['amount', 'amount_usd'] if 'amount_usd' in df_clean.columns else ['amount']
        df_clean = self.outlier_detector.detect_all_outliers(df_clean, numeric_cols)
        
        # Step 6: Handle missing values
        logging.info("Step 8/10: Handling missing values...")
        missing_analysis = self.missingness_handler.analyze_missingness(df_clean)
        for col, analysis in missing_analysis.items():
            if analysis['percentage'] > 0 and analysis['percentage'] < 80:
                df_clean = self.missingness_handler.impute_column(df_clean, col)
        
        # Step 7: Validate
        logging.info("Step 9/10: Validating cleaned data...")
        self.validation_results = self.validator.validate_all(df_clean)
        
        self.cleaned_df = df_clean
        logging.info("Step 10/10: Transformation complete.")
        
        return df_clean
    
    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """Fit and transform in one step"""
        self.fit(df)
        return self.transform(df)
    
    def get_audit_summary(self) -> Dict:
        """Get summary of all transformations"""
        return self.audit_logger.get_summary()
    
    def save_artifacts(self, output_dir: str):
        """Save all pipeline artifacts"""
        output_path = Path(output_dir)
        output_path.mkdir(parents=True, exist_ok=True)
        
        # Save cleaned data
        if self.cleaned_df is not None:
            self.cleaned_df.to_parquet(output_path / 'cleaned_data.parquet', index=False)
        
        # Save audit log
        self.audit_logger.save_audit_log(str(output_path / 'audit_log.parquet'))
        
        # Save profile
        if self.profile_results:
            with open(output_path / 'data_profile.json', 'w') as f:
                json.dump(self.profile_results, f, indent=2, default=str)
        
        # Save validation results
        if self.validation_results:
            with open(output_path / 'validation_results.json', 'w') as f:
                json.dump(self.validation_results, f, indent=2, default=str)
        
        # Save audit summary
        audit_summary = self.get_audit_summary()
        with open(output_path / 'audit_summary.json', 'w') as f:
            json.dump(audit_summary, f, indent=2)
        
        logging.info(f"All artifacts saved to {output_dir}")
    
    def generate_report(self) -> str:
        """Generate human-readable cleaning report"""
        report = []
        report.append("=" * 80)
        report.append("DATA CLEANING PIPELINE REPORT")
        report.append("=" * 80)
        report.append("")
        
        # Dataset info
        if self.original_df is not None:
            report.append(f"Original Dataset: {self.original_df.shape[0]:,} rows × {self.original_df.shape[1]} columns")
        if self.cleaned_df is not None:
            report.append(f"Cleaned Dataset: {self.cleaned_df.shape[0]:,} rows × {self.cleaned_df.shape[1]} columns")
        report.append("")
        
        # Audit summary
        audit_summary = self.get_audit_summary()
        report.append("TRANSFORMATION SUMMARY")
        report.append("-" * 80)
        report.append(f"Total Transformations: {audit_summary.get('total_transformations', 0):,}")
        report.append(f"Columns Affected: {audit_summary.get('columns_affected', 0)}")
        report.append(f"Average Confidence: {audit_summary.get('avg_confidence', 0):.2%}")
        report.append(f"Low Confidence (<0.7): {audit_summary.get('low_confidence_count', 0):,}")
        report.append("")
        
        # Rules applied
        if 'rules_applied' in audit_summary:
            report.append("RULES APPLIED")
            report.append("-" * 80)
            for rule, count in sorted(audit_summary['rules_applied'].items(), key=lambda x: x[1], reverse=True):
                report.append(f"  {rule}: {count:,}")
            report.append("")
        
        # Validation results
        if self.validation_results:
            summary = self.validation_results.get('summary', {})
            report.append("VALIDATION RESULTS")
            report.append("-" * 80)
            report.append(f"Total Checks: {summary.get('total_checks', 0)}")
            report.append(f"Passed: {summary.get('passed_checks', 0)}")
            report.append(f"Failed: {summary.get('failed_checks', 0)}")
            report.append(f"Pass Rate: {summary.get('pass_rate', 0):.1%}")
            report.append("")
        
        report.append("=" * 80)
        
        return "\n".join(report)

In [15]:
def create_synthetic_fraud_dataset(n_rows: int = 10000) -> pd.DataFrame:
    """Create synthetic fraud dataset with realistic issues"""
    np.random.seed(42)
    
    data = {
        'transaction_id': [f'TXN{i:08d}' for i in range(n_rows)],
        'timestamp': pd.date_range('2024-01-01', periods=n_rows, freq='1min'),
        'amount': np.random.lognormal(4, 1.5, n_rows),
        'currency': np.random.choice(['USD', 'EUR', 'GBP'], n_rows, p=[0.7, 0.2, 0.1]),
        'merchant_name': np.random.choice([
            'Amazon', 'Walmart', 'Target', 'BestBuy', 'Starbucks',
            'McDonalds', 'Shell', 'Exxon', 'CVS', 'Walgreens'
        ], n_rows),
        'customer_id': [f'CUST{np.random.randint(1, 1000):05d}' for _ in range(n_rows)],
        'channel': np.random.choice(['online', 'in-store', 'mobile'], n_rows, p=[0.5, 0.3, 0.2]),
        'is_fraud': np.random.choice([0, 1], n_rows, p=[0.95, 0.05])
    }
    
    df = pd.DataFrame(data)
    
    # Introduce realistic issues
    # 1. Corrupt some timestamps
    corrupt_idx = np.random.choice(df.index, size=int(0.1 * n_rows), replace=False)
    for idx in corrupt_idx:
        df.at[idx, 'timestamp'] = df.at[idx, 'timestamp'].strftime('%d/%m/%Y %H:%M:%S')
    
    # 2. Add currency symbols to amounts
    corrupt_idx = np.random.choice(df.index, size=int(0.15 * n_rows), replace=False)
    for idx in corrupt_idx:
        df.at[idx, 'amount'] = f"${df.at[idx, 'amount']:.2f}"
    
    # 3. Add typos to merchant names
    corrupt_idx = np.random.choice(df.index, size=int(0.05 * n_rows), replace=False)
    for idx in corrupt_idx:
        merchant = df.at[idx, 'merchant_name']
        df.at[idx, 'merchant_name'] = merchant + ' Inc'
    
    # 4. Introduce missing values
    for col in ['merchant_name', 'channel']:
        missing_idx = np.random.choice(df.index, size=int(0.03 * n_rows), replace=False)
        df.loc[missing_idx, col] = np.nan
    
    # 5. Create some duplicates
    dup_idx = np.random.choice(df.index, size=int(0.02 * n_rows), replace=False)
    df = pd.concat([df, df.loc[dup_idx]], ignore_index=True)
    
    # 6. Add outliers
    outlier_idx = np.random.choice(df.index, size=int(0.01 * len(df)), replace=False)
    df.loc[outlier_idx, 'amount'] = np.random.uniform(50000, 100000, len(outlier_idx))
    
    return df

In [ ]:
def run_example_pipeline():
    """Run complete example pipeline"""
    
    # Setup logging
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s'
    )
    
    print("\n" + "="*80)
    print("ENTERPRISE DATA CLEANING PIPELINE - EXAMPLE RUN")
    print("="*80 + "\n")
    
    # Create synthetic dataset
    print("Creating synthetic fraud dataset with realistic data quality issues...")
    df_raw = create_synthetic_fraud_dataset(n_rows=5000)
    print(f"Created dataset with {len(df_raw):,} rows\n")
    
    # Initialize pipeline
    print("Initializing cleaning pipeline...")
    pipeline = DataCleaningPipeline()
    
    # Run pipeline
    print("\nExecuting cleaning pipeline...\n")
    df_cleaned = pipeline.fit_transform(df_raw)
    
    # Generate report
    print("\n" + pipeline.generate_report())
    
    # Save artifacts
    output_dir = 'cleaning_artifacts'
    print(f"\nSaving artifacts to {output_dir}/...")
    pipeline.save_artifacts(output_dir)
    
    print("\n" + "="*80)
    print("PIPELINE EXECUTION COMPLETE")
    print("="*80)
    print(f"\nCleaned data shape: {df_cleaned.shape}")
    print(f"Artifacts saved to: {output_dir}/")
    print("\nGenerated files:")
    print("  - cleaned_data.parquet")
    print("  - audit_log.parquet")
    print("  - data_profile.json")
    print("  - validation_results.json")
    print("  - audit_summary.json")
    
    return pipeline, df_raw, df_cleaned


# Run the example
if __name__ == "__main__":
    pipeline, raw_data, cleaned_data = run_example_pipeline()